In [ ]:
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWai

from selenium.webdriver.support import expected_conditions as EC

URL = "https://store.steampowered.com/charts/topsellers/KR/2025-12-23"

def get_top100_appids(url: str, target=100):
    opts = Options()
    # opts.add_argument("--headless=new")  # 창 안 띄우려면 주석 해제
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--disable-gpu")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

    driver = webdriver.Chrome(options=opts)
    try:
        driver.get(url)

        wait = WebDriverWait(driver, 10)

        # 테이블(혹은 차트 루트)이 뜰 때까지 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))

        # 스크롤하면서 appid 누적 수집
        seen = []
        seen_set = set()

        def harvest():
            links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/app/']")
            for a in links:
                href = a.get_attribute("href") or ""
                m = re.search(r"/app/(\d+)/", href)
                if m:
                    appid = m.group(1)
                    if appid not in seen_set:
                        seen_set.add(appid)
                        seen.append(appid)

        # 초기 수집
        time.sleep(1)
        harvest()

        # 스크롤 루프
        last_count = -1
        same_count_rounds = 0

        while len(seen) < target and same_count_rounds < 6:
            # 페이지 아래로 스크롤 (가상리스트 렌더 유도)
            driver.execute_script("window.scrollBy(0, 900);")
            time.sleep(0.6)
            harvest()

            # 변화 감지
            if len(seen) == last_count:
                same_count_rounds += 1
            else:
                same_count_rounds = 0
                last_count = len(seen)

        return seen[:target]

    finally:
        driver.quit()

if __name__ == "__main__":
    appids = get_top100_appids(URL, target=100)
    print("count:", len(appids))
    print(appids)


ModuleNotFoundError: No module named 'selenium'

In [2]:
import time
import random
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

SECONDS_IN_DAY = 86400

# =========================
# HTTP 안정화: Session + Retry
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://store.steampowered.com/",
})

def safe_get(url, params=None, timeout=30, max_retries=8):
    """
    502/5xx, 429 등을 만나도 자동 재시도.
    """
    base_backoff = 1.5
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, params=params, timeout=timeout)

            # 429: 레이트리밋
            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                wait = int(retry_after) if retry_after and retry_after.isdigit() else 10
                wait += random.uniform(0, 1.0)
                print(f"[429] wait {wait:.1f}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                continue

            # 5xx: 서버 불안정(502 포함)
            if 500 <= r.status_code < 600:
                wait = min(60, base_backoff * (2 ** (attempt - 1))) + random.uniform(0, 0.7)
                print(f"[{r.status_code}] wait {wait:.1f}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r

        except requests.RequestException as e:
            wait = min(60, base_backoff * (2 ** (attempt - 1))) + random.uniform(0, 0.7)
            print(f"[EXC] {type(e).__name__}: {e} -> wait {wait:.1f}s (attempt {attempt}/{max_retries})")
            time.sleep(wait)

    raise RuntimeError(f"safe_get failed after {max_retries} retries: {url}")

# =========================
# 리뷰 수집 함수
# =========================
def fetch_reviews_last_n_days(
    appid: int,
    days: int = 180,
    filter: str = "recent",
    language: str = "all",
    review_type: str = "all",
    purchase_type: str = "all",
    num_per_page: int = 100,
    filter_offtopic_activity: int = 1,
    sleep_sec: float = 0.35,     # 병렬이면 너무 낮추지 말기
):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    cursor = "*"
    rows = []
    page = 0

    now_ts = int(time.time())
    cutoff_ts = now_ts - days * SECONDS_IN_DAY

    while True:
        params = {
            "json": 1,
            "filter": filter,
            "language": language,
            "review_type": review_type,
            "purchase_type": purchase_type,
            "num_per_page": num_per_page,
            "cursor": cursor,
            "filter_offtopic_activity": filter_offtopic_activity,
        }

        r = safe_get(url, params=params, timeout=30, max_retries=8)
        data = r.json()

        if data.get("success") != 1:
            raise RuntimeError(f"API success != 1: {data}")

        reviews = data.get("reviews", [])
        if not reviews:
            break

        stop = False
        for rev in reviews:
            ts_created = rev.get("timestamp_created")

            # 기간 컷
            if ts_created is not None and ts_created < cutoff_ts:
                stop = True
                break

            author = rev.get("author", {})
            rows.append({
                "appid": str(appid),
                "recommendationid": rev.get("recommendationid"),
                "steamid": author.get("steamid"),
                "num_games_owned": author.get("num_games_owned"),
                "num_reviews_author": author.get("num_reviews"),
                "playtime_forever": author.get("playtime_forever"),
                "playtime_last_two_weeks": author.get("playtime_last_two_weeks"),
                "playtime_at_review": author.get("playtime_at_review"),
                "deck_playtime_at_review": author.get("deck_playtime_at_review"),
                "last_played": author.get("last_played"),
                "language": rev.get("language"),
                "review": rev.get("review"),
                "timestamp_created": ts_created,
                "timestamp_updated": rev.get("timestamp_updated"),
                "voted_up": rev.get("voted_up"),
                "votes_up": rev.get("votes_up"),
                "votes_funny": rev.get("votes_funny"),
                "weighted_vote_score": rev.get("weighted_vote_score"),
                "comment_count": rev.get("comment_count"),
                "steam_purchase": rev.get("steam_purchase"),
                "received_for_free": rev.get("received_for_free"),
                "written_during_early_access": rev.get("written_during_early_access"),
                "developer_response": rev.get("developer_response"),
                "timestamp_dev_responded": rev.get("timestamp_dev_responded"),
                "primarily_steam_deck": rev.get("primarily_steam_deck"),
            })

        page += 1
        print(f"appid={appid}, page={page}, fetched={len(reviews)}, kept_total={len(rows)}")

        if stop:
            break

        cursor = data.get("cursor")
        time.sleep(sleep_sec)

    return pd.DataFrame(rows)

# =========================
# 병렬 실행 + 실패 1회 재시도
# =========================
def run_batch(appids, output_path, days=180, max_workers=4):
    """
    appid들을 병렬로 처리하고,
    각 appid 결과는 즉시 CSV append 저장.
    반환: 실패한 appid 리스트
    """
    first_write = not pd.io.common.file_exists(output_path)
    failed = []

    def job(appid):
        df = fetch_reviews_last_n_days(int(appid), days=days)
        return str(appid), df

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(job, appid): appid for appid in appids}

        for fut in as_completed(futures):
            appid = futures[fut]
            try:
                appid_done, df = fut.result()

                # 결과가 비어도 appid는 성공 처리 (원하면 빈 것도 실패로 바꿀 수 있음)
                df.to_csv(
                    output_path,
                    mode="w" if first_write else "a",
                    header=first_write,
                    index=False,
                    encoding="utf-8-sig"
                )
                first_write = False
                print(f"done {appid_done}: {df.shape}")

                # 병렬이니까 너무 공격적으로 추가요청 안 하게 약간 랜덤 쉬기
                time.sleep(random.uniform(0.2, 0.6))

            except Exception as e:
                print(f"fail {appid}: {e}")
                failed.append(str(appid))

    return failed

# =========================
# 실행
# =========================
app_id_list = [
    '730', '1049590', '578080', '3564740', '3557620', '2139460', '1808500', '960170',
    '2344520', '2879840', '1245620', '1086940', '1771300', '1091500', '2246340',
    '3513350', '3240220', '985810', '1903340', '2426960', '1623730', '236390',
    '3527290', '216150', '2827200', '381210', '2807960', '413150', '261550',
    '2622380', '1426210', '1973530', '3609080', '294100', '1174180', '1172470',
    '2444750', '648800', '3241660', '2592160', '2001120', '3489700', '3551340',
    '3405690', '3932890', '728880', '553850', '1984270', '3159330', '4077430',
    '108600', '440', '990080', '1145350', '2138330', '230410', '1621690',
    '3167020', '3472040', '394360', '1326470', '1449850', '1158310', '1144200',
    '1222140', '2215430', '227300', '3059520', '1030300', '2993780', '3101040',
    '1222670', '2927200', '934700', '570', '286160', '835570', '814380',
    '1778820', '1260320', '1142710', '1825750', '526870', '1435790', '1562700',
    '3447040', '1868140', '1627720', '2651280', '1203620', '1966720', '2183900',
    '2948190', '1929290', '2436940', '2968420', '2322010', '703080', '1551360',
    '1888160'
]

if __name__ == "__main__":
    OUTPUT = "steam_reviews_last180d.csv"

    print("=== 1st pass (parallel) ===")
    failed_1 = run_batch(app_id_list, OUTPUT, days=180, max_workers=4)
    print("1st failed:", failed_1)

    # 실패 리스트는 마지막에 딱 1번만 더 시도
    if failed_1:
        print("\n=== retry failed once ===")
        time.sleep(10)  # 재시도 전 텀 조금
        failed_2 = run_batch(failed_1, OUTPUT, days=180, max_workers=2)  # 재시도는 더 보수적으로
        print("retry failed:", failed_2)

    print("\nDONE.")


=== 1st pass (parallel) ===
appid=730, page=1, fetched=99, kept_total=99
appid=578080, page=1, fetched=100, kept_total=100
appid=1049590, page=1, fetched=100, kept_total=100
appid=3564740, page=1, fetched=100, kept_total=100
appid=1049590, page=2, fetched=100, kept_total=200
appid=730, page=2, fetched=100, kept_total=199
appid=578080, page=2, fetched=100, kept_total=200
appid=3564740, page=2, fetched=100, kept_total=200
appid=1049590, page=3, fetched=100, kept_total=300
appid=730, page=3, fetched=100, kept_total=299
appid=3564740, page=3, fetched=100, kept_total=300
appid=1049590, page=4, fetched=100, kept_total=400
appid=730, page=4, fetched=100, kept_total=399
appid=3564740, page=4, fetched=100, kept_total=400
appid=1049590, page=5, fetched=100, kept_total=500
appid=578080, page=3, fetched=100, kept_total=300
appid=730, page=5, fetched=100, kept_total=499
appid=730, page=6, fetched=100, kept_total=599
appid=1049590, page=6, fetched=100, kept_total=600
appid=3564740, page=5, fetched=1

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0"}

def steamcharts_monthly(appid: int, months: int = 6) -> pd.DataFrame:
    """
    SteamCharts 앱 페이지에서 월별 테이블(월, 평균, 피크, 증감%)을 파싱.
    months: 최근 몇 개월을 가져올지 (기본 6)
    """
    url = f"https://steamcharts.com/app/730"
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    table = soup.find("table", class_="common-table")
    if table is None:
        raise ValueError("monthly table(common-table)을 못 찾았어요. 페이지 구조가 바뀌었을 수 있어요.")

    rows = table.find_all("tr")[1:1+months]  # 헤더 제외 후 최근 months개
    out = []
    for tr in rows:
        tds = tr.find_all("td")
        # 보통: Month | Avg. Players | Gain | % Gain | Peak Players
        month = tds[0].get_text(strip=True)
        avg = tds[1].get_text(strip=True).replace(",", "")
        gain = tds[2].get_text(strip=True).replace(",", "")
        pct = tds[3].get_text(strip=True)
        peak = tds[4].get_text(strip=True).replace(",", "")

        def to_float(x):
            try: return float(x)
            except: return None

        out.append({
            "appid": appid,
            "month": month,
            "avg_players": to_float(avg),
            "gain": to_float(gain),
            "pct_gain": pct,      # "+1.23%" 같은 문자열 유지
            "peak_players": to_float(peak),
        })

    return pd.DataFrame(out)


df6 = steamcharts_monthly(730, months=6)  # CS2/CSGO appid 예시
print(df6)

   appid           month  avg_players      gain pct_gain  peak_players
0    730    Last 30 Days   1025542.43  34365.50   +3.47%     1620137.0
1    730   December 2025    991176.89   9694.44   +0.99%     1620137.0
2    730   November 2025    981482.45  33238.24   +3.51%     1597285.0
3    730    October 2025    948244.21  22303.90   +2.41%     1588091.0
4    730  September 2025    925940.31  -2488.93   -0.27%     1571060.0
5    730     August 2025    928429.24   8359.26   +0.91%     1502245.0


In [5]:
import numpy as np

def make_6m_features(df_monthly: pd.DataFrame) -> dict:
    vals = df_monthly["avg_players"].to_numpy(dtype=float)
    first, last = vals[0], vals[-1]

    return {
        "avg_players_6m": float(np.nanmean(vals)),
        "trend_6m": float(last - first),
        "drop_ratio_6m": float((first - last) / first) if first and first > 0 else 0.0,
        "volatility_6m": float(np.nanstd(vals)),
    }
    
make_6m_features(df6)

{'avg_players_6m': 966802.5883333334,
 'trend_6m': -97113.19000000006,
 'drop_ratio_6m': 0.0946944632997779,
 'volatility_6m': 35931.942564104174}

In [54]:
import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0"}

# -----------------------------
# 1) Store: appdetails
# -----------------------------
def fetch_appdetails(appid: int, cc="us", lang="english") -> dict:
    url = "https://store.steampowered.com/api/appdetails"
    r = requests.get(url, params={"appids": appid, "cc": cc, "l": lang}, headers=HEADERS, timeout=20)
    r.raise_for_status()
    j = r.json()

    payload = j.get(str(appid), {})
    if not payload.get("success"):
        return {}

    data = payload.get("data", {}) or {}

    required_age = data.get("required_age")

    # price: 보기 좋게 "initial/final/discount currency" 한 칸 문자열로 만들기
    price = None
    po = data.get("price_overview") or None
    if po:
        # store API는 종종 센트 단위(예: 19900)로 내려옴
        # 여기서는 그대로 숫자값을 사용하고, 한 칸 문자열로 합침
        initial = po.get("initial")
        final = po.get("final")
        disc = po.get("discount_percent")
        cur = po.get("currency")
        parts = []
        if initial is not None: parts.append(f"initial={initial}")
        if final is not None: parts.append(f"final={final}")
        if disc is not None: parts.append(f"discount%={disc}")
        if cur: parts.append(f"currency={cur}")
        price = " ".join(parts) if parts else None
    else:
        # 무료게임 등
        if data.get("is_free") is True:
            price = "FREE"

    metacritic_score = (data.get("metacritic") or {}).get("score")

    achievements_total = (data.get("achievements") or {}).get("total")

    supported_languages = data.get("supported_languages")
    # 옵션: HTML 태그 제거 (<br> 등)
    if isinstance(supported_languages, str):
        supported_languages = re.sub(r"<.*?>", " ", supported_languages)
        supported_languages = re.sub(r"\s+", " ", supported_languages).strip()

    genres = data.get("genres") or []
    genre = [g.get("description") for g in genres if isinstance(g, dict) and g.get("description")]

    return {
        "required_age": required_age,
        "price": price,
        "metacritic_score": metacritic_score,
        "achievements": achievements_total,
        "supported_languages": supported_languages,
        "genre": genre,
    }

# -----------------------------
# 2) Store: appreviews (total / recent)
# -----------------------------
def fetch_review_summary(appid: int, filter_: str) -> dict:
    url = f"https://store.steampowered.com/appreviews/{appid}"
    r = requests.get(
        url,
        params={"json": 1, "language": "all", "filter": filter_},
        headers=HEADERS,
        timeout=20,
    )
    r.raise_for_status()
    j = r.json()

    qs = j.get("query_summary", {}) or {}
    pos = qs.get("total_positive") or 0
    neg = qs.get("total_negative") or 0
    total = pos + neg
    pct_pos = (pos / total * 100.0) if total > 0 else None

    return {
        "pct_pos": pct_pos,
        "num_reviews": qs.get("total_reviews"),
        "review_score_desc": qs.get("review_score_desc"),
    }

# -----------------------------
# 3) SteamCharts: peak ccu (All-time peak)
# -----------------------------
def fetch_steamcharts_peak_ccu(appid: int):
    url = f"https://steamcharts.com/app/{appid}"
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # 페이지에 보통 "All-time Peak"가 텍스트로 있고, 숫자는 그 근처에 있음
    # 구조 변화에 대비해서: "All-time Peak" 문구를 포함한 텍스트에서 숫자만 추출
    text = soup.get_text(" ", strip=True)

    m = re.search(r"All-time Peak\s*([\d,]+)", text)
    if not m:
        # 예비 패턴
        m = re.search(r"All-time Peak\s*:\s*([\d,]+)", text)

    if not m:
        return None

    return int(m.group(1).replace(",", ""))

# -----------------------------
# 4) 원하는 컬럼만 "딱" 생성 + CSV 저장
# -----------------------------
FINAL_COLS = [
    "appid",
    "required_age",
    "price",
    "metacritic_score",
    "achievements",
    "supported_languages",
    "user_score",
    "peak_ccu",
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent",
    "genre",
]

def build_one_row(appid: int, cc="us", lang="english", sleep_sec=0.2) -> dict:
    appid = int(appid)

    # appdetails
    d = fetch_appdetails(appid, cc=cc, lang=lang)

    # reviews (total/recent)
    total = fetch_review_summary(appid, "all")
    recent = fetch_review_summary(appid, "recent")

    # peak ccu
    peak = fetch_steamcharts_peak_ccu(appid)

    # user_score는 너가 “추천 평가 지표”로 쓰고 싶은 값으로 정의하면 됨.
    # 여기서는 "전체 긍정비율(pct_pos_total)"을 user_score로 둠.
    row = {
        "appid": appid,
        "required_age": d.get("required_age"),
        "price": d.get("price"),
        "metacritic_score": d.get("metacritic_score"),
        "achievements": d.get("achievements"),
        "supported_languages": d.get("supported_languages"),
        "user_score": total.get("pct_pos"),               # <= 너 정의
        #"peak_ccu": recent.get("peak_ccu"),
        "pct_pos_total": total.get("pct_pos"),
        "num_reviews_total": total.get("num_reviews"),
        "pct_pos_recent": recent.get("pct_pos"),
        "num_reviews_recent": recent.get("num_reviews"),
        "genre": d.get("genre"),
    }

    if sleep_sec:
        time.sleep(sleep_sec)

    # 혹시라도 키가 더 생기지 않도록 최종 컬럼만 남김
    return {k: row.get(k) for k in FINAL_COLS}

def fetch_and_save(appids, out_csv="steam_game_meta_min.csv", cc="us", lang="english", sleep_sec=0.2, max_apps=None):
    appids = [int(a) for a in appids if pd.notna(a)]
    # 중복 제거(순서 유지)
    seen = set()
    appids = [a for a in appids if (a not in seen and not seen.add(a))]

    if max_apps is not None:
        appids = appids[:max_apps]

    rows = []
    for i, appid in enumerate(appids, 1):
        try:
            rows.append(build_one_row(appid, cc=cc, lang=lang, sleep_sec=sleep_sec))
        except Exception as e:
            # 실패해도 컬럼은 유지
            rows.append({k: (appid if k == "appid" else None) for k in FINAL_COLS})
            print(f"[warn] appid={appid} failed: {e}")

        if i % 50 == 0:
            print(f"[progress] {i}/{len(appids)}")

    df = pd.DataFrame(rows, columns=FINAL_COLS)  # 컬럼 순서 강제
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[saved] {out_csv} rows={len(df)} cols={df.shape[1]}")
    return df

if __name__ == "__main__":
    appids = app_id_list = [
    '730', '1049590', '578080',
    '3564740', '3557620', '2139460', '1808500', '960170', '2344520', '2879840', '1245620', '1086940', '1771300', '1091500', '2246340',
    '3513350', '3240220', '985810', '1903340', '2426960', '1623730', '236390',
    '3527290', '216150', '2827200', '381210', '2807960', '413150', '261550',
    '2622380', '1426210', '1973530', '3609080', '294100', '1174180', '1172470',
    '2444750', '648800', '3241660', '2592160', '2001120', '3489700', '3551340',
    '3405690', '3932890', '728880', '553850', '1984270', '3159330', '4077430',
    '108600', '440', '990080', '1145350', '2138330', '230410', '1621690',
    '3167020', '3472040', '394360', '1326470', '1449850', '1158310', '1144200',
    '1222140', '2215430', '227300', '3059520', '1030300', '2993780', '3101040',
    '1222670', '2927200', '934700', '570', '286160', '835570', '814380',
    '1778820', '1260320', '1142710', '1825750', '526870', '1435790', '1562700',
    '3447040', '1868140', '1627720', '2651280', '1203620', '1966720', '2183900',
    '2948190', '1929290', '2436940', '2968420', '2322010', '703080', '1551360',
    '1888160'
]
fetch_and_save(appids, out_csv="steam_game_meta_info.csv", sleep_sec=0.3)


[progress] 50/100
[progress] 100/100
[saved] steam_game_meta_info.csv rows=100 cols=13


,appid,required_age,price,metacritic_score,achievements,supported_languages,user_score,peak_ccu,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent,genre
0,730,0,FREE,NaN,1.0,"Czech, Danish, Dutch, English * , Finnish, Fre...",85.694959,None,85.694959,4859224,85.694959,4859224,"[Action, Free To Play]"
1,1049590,0,FREE,NaN,NaN,"English * , Japanese * , Simplified Chinese, K...",81.308411,None,81.308411,107,81.308411,107,"[Indie, Strategy, Free To Play]"
2,578080,13,FREE,NaN,37.0,"English, Korean, Simplified Chinese, French, G...",55.724266,None,55.724266,1754094,55.724266,1754094,"[Action, Adventure, Massively Multiplayer, Fre..."
3,3564740,0,FREE,NaN,34.0,"English * , Japanese, Simplified Chinese * , T...",NaN,None,NaN,0,NaN,0,"[Action, Adventure, RPG, Free To Play]"
4,3557620,0,FREE,NaN,NaN,"English, Traditional Chinese, Thai, Korean * *...",NaN,None,NaN,0,NaN,0,"[Adventure, Casual, RPG, Strategy, Free To Play]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2968420,0,initial=2499 final=2499 discount%=0 currency=USD,NaN,40.0,"English, French, Italian, German, Spanish - Sp...",92.203717,None,92.203717,8286,92.203717,8286,"[Casual, Indie, Simulation]"
96,2322010,17,initial=5999 final=5999 discount%=0 currency=USD,90.0,48.0,"English * , French * , Italian * , German * , ...",88.570215,None,88.570215,25906,88.570215,25906,"[Action, Adventure, RPG]"
97,703080,0,initial=4499 final=4499 discount%=0 currency=USD,81.0,38.0,"English * , French * , German * , Spanish - Sp...",91.009593,None,91.009593,76615,91.009593,76615,"[Casual, Simulation, Strategy]"
98,1551360,0,initial=5999 final=5999 discount%=0 currency=USD,NaN,164.0,"English * , French * , Italian * , German * , ...",89.280232,None,89.280232,238410,89.280232,238410,"[Action, Adventure, Racing, Simulation, Sports]"


In [56]:
dfdf = pd.read_csv('data/steam_game_meta_info.csv')
dfdf['price'].unique()

array(['FREE', 'initial=3999 final=3999 discount%=0 currency=USD',
       'initial=4999 final=4999 discount%=0 currency=USD',
       'initial=1999 final=1999 discount%=0 currency=USD',
       'initial=5999 final=5999 discount%=0 currency=USD',
       'initial=6999 final=6999 discount%=0 currency=USD',
       'initial=2999 final=2999 discount%=0 currency=USD',
       'initial=799 final=495 discount%=38 currency=USD',
       'initial=999 final=999 discount%=0 currency=USD',
       'initial=6999 final=4899 discount%=30 currency=USD',
       'initial=1499 final=1499 discount%=0 currency=USD',
       'initial=3999 final=799 discount%=80 currency=USD', nan,
       'initial=3499 final=3499 discount%=0 currency=USD',
       'initial=2499 final=2499 discount%=0 currency=USD',
       'initial=999 final=699 discount%=30 currency=USD',
       'initial=5999 final=5399 discount%=10 currency=USD',
       'initial=1799 final=1799 discount%=0 currency=USD',
       'initial=5999 final=2999 discount%=50 

In [57]:
import numpy as np
import pandas as pd

def extract_initial_price(x):
    if pd.isna(x):
        return np.nan
    if x == "FREE":
        return "FREE"
    if isinstance(x, str) and x.startswith("initial="):
        # initial=숫자 부분만 추출
        return int(x.split("initial=")[1].split()[0])
    return np.nan  # 예외 케이스 안전장치

dfdf["price"] = dfdf["price"].apply(extract_initial_price)

dfdf

,appid,required_age,price,metacritic_score,achievements,supported_languages,user_score,peak_ccu,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent,genre
0,730,0.0,FREE,NaN,1.0,"Czech, Danish, Dutch, English * , Finnish, Fre...",85.694959,NaN,85.694959,4859224,85.694959,4859224,"['Action', 'Free To Play']"
1,1049590,0.0,FREE,NaN,NaN,"English * , Japanese * , Simplified Chinese, K...",81.308411,NaN,81.308411,107,81.308411,107,"['Indie', 'Strategy', 'Free To Play']"
2,578080,13.0,FREE,NaN,37.0,"English, Korean, Simplified Chinese, French, G...",55.724266,NaN,55.724266,1754094,55.724266,1754094,"['Action', 'Adventure', 'Massively Multiplayer..."
3,3564740,0.0,FREE,NaN,34.0,"English * , Japanese, Simplified Chinese * , T...",NaN,NaN,NaN,0,NaN,0,"['Action', 'Adventure', 'RPG', 'Free To Play']"
4,3557620,0.0,FREE,NaN,NaN,"English, Traditional Chinese, Thai, Korean * *...",NaN,NaN,NaN,0,NaN,0,"['Adventure', 'Casual', 'RPG', 'Strategy', 'Fr..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2968420,0.0,2499,NaN,40.0,"English, French, Italian, German, Spanish - Sp...",92.203717,NaN,92.203717,8286,92.203717,8286,"['Casual', 'Indie', 'Simulation']"
96,2322010,17.0,5999,90.0,48.0,"English * , French * , Italian * , German * , ...",88.570215,NaN,88.570215,25906,88.570215,25906,"['Action', 'Adventure', 'RPG']"
97,703080,0.0,4499,81.0,38.0,"English * , French * , German * , Spanish - Sp...",91.009593,NaN,91.009593,76615,91.009593,76615,"['Casual', 'Simulation', 'Strategy']"
98,1551360,0.0,5999,NaN,164.0,"English * , French * , Italian * , German * , ...",89.280232,NaN,89.280232,238410,89.280232,238410,"['Action', 'Adventure', 'Racing', 'Simulation'..."


In [58]:
dfdf.isnull().sum()

appid                    0
required_age             2
price                    2
metacritic_score        63
achievements            20
supported_languages      2
user_score              10
peak_ccu               100
pct_pos_total           10
num_reviews_total        0
pct_pos_recent          10
num_reviews_recent       0
genre                    2
dtype: int64

In [59]:
dfdf.isnull().sum()

appid                    0
required_age             2
price                    2
metacritic_score        63
achievements            20
supported_languages      2
user_score              10
peak_ccu               100
pct_pos_total           10
num_reviews_total        0
pct_pos_recent          10
num_reviews_recent       0
genre                    2
dtype: int64

In [60]:
dfdf["required_age"] = dfdf["required_age"].fillna(0)


In [61]:
dfdf["price_missing"] = dfdf["price"].isna().astype(int)
dfdf["price"] = dfdf["price"].fillna(0)

In [62]:
dfdf["metacritic_missing"] = dfdf["metacritic_score"].isna().astype(int)
dfdf["metacritic_score"] = dfdf["metacritic_score"].fillna(dfdf["metacritic_score"].median())

In [63]:
dfdf["achievements"] = dfdf["achievements"].fillna(0)


In [64]:
dfdf["supported_languages"] = dfdf["supported_languages"].fillna("unknown")

In [65]:
dfdf["user_score_missing"] = dfdf["user_score"].isna().astype(int)
dfdf["user_score"] = dfdf["user_score"].fillna(dfdf["user_score"].median())

In [66]:
dfdf["pct_pos_total"] = dfdf["pct_pos_total"].fillna(dfdf["pct_pos_total"].median())


In [67]:
dfdf["pct_pos_recent_missing"] = dfdf["pct_pos_recent"].isna().astype(int)
dfdf["pct_pos_recent"] = dfdf["pct_pos_recent"].fillna(dfdf["pct_pos_recent"].median())

In [68]:
dfdf["genre"] = dfdf["genre"].fillna("unknown")

In [69]:
dfdf.isnull().sum()

appid                       0
required_age                0
price                       0
metacritic_score            0
achievements                0
supported_languages         0
user_score                  0
peak_ccu                  100
pct_pos_total               0
num_reviews_total           0
pct_pos_recent              0
num_reviews_recent          0
genre                       0
price_missing               0
metacritic_missing          0
user_score_missing          0
pct_pos_recent_missing      0
dtype: int64

In [70]:
dfdf = dfdf.drop(columns=['peak_ccu'])
dfdf.isnull().sum()

appid                     0
required_age              0
price                     0
metacritic_score          0
achievements              0
supported_languages       0
user_score                0
pct_pos_total             0
num_reviews_total         0
pct_pos_recent            0
num_reviews_recent        0
genre                     0
price_missing             0
metacritic_missing        0
user_score_missing        0
pct_pos_recent_missing    0
dtype: int64

In [71]:
dfdf.to_csv("steam_game_info.csv", index=False, encoding="utf-8-sig")


In [73]:
asd = pd.read_csv('data/steam_reviews_last180d.csv')

asd['appid'].nunique()

C:\Users\anajr\AppData\Local\Temp\ipykernel_34544\2823260827.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  asd = pd.read_csv('data/steam_reviews_last180d.csv')


100